# Load libraries and functions

In [ ]:
suppressPackageStartupMessages({
library(Seurat)
library(scCustomize)
library(spatula)
library(ComplexHeatmap)
    library(rcna)
    library(circlize)
    library(ggrastr)
    library(tidyverse)
    library(scico)
    library(circlize)
    library(ggsci)
    })

In [ ]:
library(ggrastr)
library(dplyr)


In [ ]:
get_theme <- function(size=12, angle=45) {
    defined_theme = theme_bw(base_size=size) + theme(legend.title=element_blank(), strip.text=element_text(size=size), legend.text=element_text(size=size), axis.title.x=element_text(size=size), axis.title.y=element_text(size=size), axis.text.y=element_text(size=size), axis.text.x=element_text(size=size, angle=angle, hjust = 1, vjust=1.05), legend.position="bottom", legend.box = "horizontal")
    defined_theme
}

fig.size <- function(h, w) {
    options(repr.plot.height = h, repr.plot.width = w)
}

# Configure colors

In [ ]:
heatmap_col_fun <- colorRamp2(c(-2, 0, 2), scico(3, palette = "vik"))  # "vik" is diverging

niche_cols = pal_npg("nrc")(9)
niche_cols

In [ ]:
celltype_colors <- c(
  # T/ILC
  "T Cell"                    = "#a1d99b",
  "Regulatory T Cell"         = "#74c476",
  "Proliferating T Cell"      = "#31a354",
  "Immune Cell (LowQ)"        = "#e5f5e0",

  # B/Plasma
  "B Cell"                    = "#bcbddc",
  "Plasma Cell"               = "#756bb1",

  # Myeloid
  "Tissue Myeloid"            = "#fff7bc",
  "Monocyte"                  = "#fec44f",
  "Inflammatory Myeloid"      = "#fe9929",
  "cDC1"                      = "#ec7014",
  "pDC"                       = "#cc4c02",
  "Mast Cell"                 = "#993404",

  # Endothelial / Stromal
  "Endothelial Cell"          = "#6baed6",
  "Stromal Cell"              = "#67000d",

  # Kidney epithelial
  "Podocyte"                  = "#fcbba1",
  "Parietal Cell"                  = "#fc9272",
  "Proximal Tubule"           = "#fb6a4a",
  "Loop of Henle"             = "#a50f15",
  "Distal Convoluted Tubule"  = "#9e9ac8",
  "Collecting Duct"           = "#807dba"
)

# Configure arrow

In [ ]:
axis <- ggh4x::guide_axis_truncated(
  trunc_lower = unit(0, "npc"),
  trunc_upper = unit(3, "cm")
)


# Load data

In [ ]:
sc.niche <- readRDS("sopa_baysor_tessera.rds")
lennard.subtype <- readRDS("250721_cells_annotated_lennard.rds")
imm.niche <- readRDS("250711_niches.rds")

In [ ]:
niche.merge = imm.niche 
obj.merge = lennard.subtype
orig.merge = readRDS("all_KPMP_integrate_singlet_umap_umapnn_labels_umap.rds")

In [ ]:
meta = read.csv("shruti_meta_clean (3).csv")
input_meta = meta[,c('slide_id', 'age', 'sex', 'case_ctrl', 'ICPi',  'malignancy', 'eGFR_base')] %>% arrange(case_ctrl)

In [ ]:
input_meta

## Check data

In [ ]:
cells_to_keep <- colnames(orig.merge)[orig.merge$tech=='xenium']
orig.merge.xen <- subset(orig.merge, cells = cells_to_keep)

# Main cells
main_cells <- colnames(orig.merge.xen)

# Assay cells
assay_cells <- colnames(orig.merge.xen@assays$RNA)  # or whichever assay you're using

# PCA cells
pca_cells <- rownames(orig.merge.xen@reductions$pca@cell.embeddings)

# Graph cells
graph_cells <- colnames(orig.merge.xen[['humap_fgraph']])  # adapt if using another graph

# Active identity names
ident_cells <- names(Idents(orig.merge.xen))

# Check mismatches
length(setdiff(assay_cells, main_cells))
length(setdiff(pca_cells, main_cells))
length(setdiff(graph_cells, main_cells))
length(setdiff(ident_cells, main_cells))

## Clean meta data in Seurat object

In [ ]:
orig.merge.xen@meta.data  = orig.merge.xen@meta.data %>% mutate(sample_id=str_extract(sample, "__(BS\\d*[_-].*)__2024", group=1))
orig.merge.xen@meta.data  = orig.merge.xen@meta.data %>% mutate(case_ctrl=str_trim(input_meta[match(sample_id, input_meta$slide_id), 'case_ctrl']))
orig.merge.xen@meta.data$case_ctrl_num = as.numeric(factor(str_trim(orig.merge.xen@meta.data$case_ctrl), levels=c("Control", "Case")))

In [ ]:
obj.merge@meta.data  = obj.merge@meta.data %>% mutate(sample_id=str_extract(sample, "__(BS\\d*[_-].*)__2024", group=1))
obj.merge@meta.data  = obj.merge@meta.data %>% mutate(case_ctrl=str_trim(input_meta[match(sample_id, input_meta$slide_id), 'case_ctrl']))
obj.merge@meta.data$case_ctrl_num = as.numeric(factor(str_trim(obj.merge@meta.data$case_ctrl), levels=c("Control", "Case")))
obj.merge@meta.data$cell_label = gsub(" Cell", "", obj.merge@meta.data$lennard_label)
obj.merge@meta.data = obj.merge@meta.data %>% mutate(cell_label = ifelse(cell_label=='Immune', 'Immune (LowQ)', cell_label))


In [ ]:
orig.baysor <- readRDS("kidney_orig_seg_merged.rds")
orig.baysor@meta.data <- orig.baysor@meta.data%>%unite("uniq_id", c(sample, cell_id), remove=F)
lennard.subtype@meta.data <- lennard.subtype@meta.data%>%unite("uniq_id", c(sample, cell_id), remove=F)
xy <- Embeddings(orig.baysor, 'spatial')[match(lennard.subtype@meta.data$uniq_id, orig.baysor@meta.data$uniq_id),]
rm(orig.baysor)

# fig 1b

In [ ]:
# markers = c("CD8A", "CD2", "ZAP70", "CTLA4", "FOXP3", "TIGIT", "CD3E", "MKI67", "TUBB", "XBP1", "CD38", "FCRL5", "MS4A1", 
#             "CD19", "CD79A", "CD163", "MRC1", "F13A1", "CD14", "CIITA", "FCN1", "CXCL9", "CXCL10", "MMP9", "WDFY4", "CLEC9A", 
#             "IRF8", "GZMB", "LILRA4", "IL3RA", "KIT", "HDC",  "PECAM1", "PLVAP", "PDGFRB", "PODXL", "FGF1", "NES", "SHANK3",  
#             "ITGB3", "CFH", "BMP7", "TNC", "AEBP1", "LRP2", "CUBN", "PAH", "ITGB6", "CA12", "MUC1",
#             "EPCAM", "PROM1", "PAX8", "HSD11B2", "KCNJ10", "SERPINA5", "UMOD", "CASR", "SCNN1A", "SLC4A1", 
#             "DMRT2", "CLNK", "SCNN1G", "GATA3", "PFKFB3", "PKHD1", "CALB1", "KCNJ1")

In [ ]:
# obj.merge@meta.data  = obj.merge@meta.data %>% mutate(cell_label=case_when(
#     cell_label %in% c("Basophil") ~ "Mast", 
#     cell_label %in% c("Interstitial") ~ "Stroma",  
#     .default = cell_label 
# )
#                                                      )

# obj.merge@meta.data  = obj.merge@meta.data %>% mutate(cell_label=case_when(
#     cell_label %in% c("Collecting Duct-PC", "Collecting Duct-IC", "Connecting Tubule") ~ "Collecting Duct", 
#     cell_label %in% c("Thin Descending Limb", "Thick Ascending Limb", "Thin Ascending Limb") ~ "Loop of Henle",  
# ###Thin Descending Limb”, “Thick Ascending Limb” and “Thin Ascending Limb” into one cell type “Loop of Henle    
#     .default = cell_label 
# ))

In [ ]:
markers = c("CD8A", "CD3E", "ZAP70", "CTLA4", "FOXP3", "TIGIT", "MKI67", "LMNB1", "TOP2A", "MS4A1", "CD19", "CD79A", "XBP1", "CD38", "FCRL5",  "CD163", "MRC1",  "F13A1", "CD14", "CIITA", "FCN1", "CXCL9", "CXCL10", "MMP9", "WDFY4", "CLEC9A", "IRF8", "GZMB", "LILRA4", "IL3RA", "KIT", "HDC", "PECAM1", "PLVAP", "SHANK3", "PODXL", "FGF1", "NES", "ITGB3",  "BMP7", "LRP2", "CUBN", "PAH", "UMOD",
  "SIM2", "IRX1", "PDGFRB", "CFH", "AEBP1", "CALB1", "KCNJ10", "TRPM6", "KCNJ15", "SCNN1G", "CDH1")

In [ ]:
library(dplyr)
library(stringr)

obj.merge@meta.data <- obj.merge@meta.data %>%
  mutate(
    cell_label = str_trim(as.character(cell_label)),

    cell_label = case_when(
      is.na(cell_label) ~ NA_character_,

      # Kidney epithelial cell type collapsing
      cell_label %in% c(
        "Collecting Duct-PC",
        "Collecting Duct-IC",
        "Connecting Tubule"
      ) ~ "Collecting Duct",

      cell_label %in% c(
        "Thin Descending Limb",
        "Thick Ascending Limb",
        "Thin Ascending Limb"
      ) ~ "Loop of Henle",

      # Restore "Cell" suffix
      cell_label == "T" ~ "T Cell",
      cell_label == "T cell" ~ "T Cell",

      cell_label == "Regulatory T" ~ "Regulatory T Cell",
      cell_label == "Regulatory T cell" ~ "Regulatory T Cell",

      cell_label == "Proliferating T cell" ~ "Proliferating T Cell",
      cell_label == "Proliferating T" ~ "Proliferating T Cell",

      cell_label == "Immune (LowQ)" ~ "Immune Cell (LowQ)",

      cell_label == "Plasma" ~ "Plasma Cell",
      cell_label == "Plasma cell" ~ "Plasma Cell",

      cell_label == "B" ~ "B Cell",
      cell_label == "B cell" ~ "B Cell",

      cell_label %in% c("Mast", "Mast cell", "Basophil") ~ "Mast Cell",

      cell_label == "Endothelial" ~ "Endothelial Cell",
      cell_label == "Endothelial cell" ~ "Endothelial Cell",

      cell_label == "Parietal" ~ "Parietal Cell",

      cell_label %in% c("Stroma", "Stromal", "Interstitial") ~ "Stromal Cell",

      .default = cell_label
    )
  )

In [ ]:
missing_colors <- setdiff(
  sort(unique(obj.merge$cell_label)),
  names(celltype_colors)
)

missing_colors

In [ ]:
obj.merge@meta.data = obj.merge@meta.data %>% mutate(cell_label = factor(cell_label, 
                                                                         levels=names(celltype_colors)))

In [ ]:
sum(is.na(obj.merge@meta.data$cell_label))

In [ ]:
fig1b = DotPlot_scCustom(obj.merge, features=markers, group.by='cell_label') + get_theme(size=11, angle=45)

In [ ]:
fig1b

# Fig. 1f

In [ ]:
length(names(celltype_colors))

In [ ]:
fig.size(5,5)
p.bar1 <- obj.merge@meta.data %>% count(case_ctrl, cell_label) %>% group_by(case_ctrl) %>% mutate(ratio=n/sum(n)) %>% mutate(cell_label = factor(cell_label, levels=names(celltype_colors))) %>% 
    ggplot() + geom_bar(aes(x=case_ctrl, y=ratio, fill=cell_label), stat='identity') + scale_fill_manual(values=celltype_colors) + theme_bw() + theme(
#    legend.title = element_text(size = 6),  # Legend title size
#    legend.text  = element_text(size = 5),  # Legend labels size
#    legend.key.size = unit(0.4, "cm"),
    axis.title.y = element_text(size = 15),  # y-axis title font size
    axis.text.y  = element_text(size = 13),  # y-axis tick labels font size        
    axis.text.x = element_text(size=15, angle = 45, hjust = 1, vjust=1.05),
    legend.position='none',
    plot.title = element_text(
      size = 16,      # font size
      face = "bold",  # "plain", "italic", "bold", "bold.italic"
      hjust = 0.5     # 0 = left, 0.5 = center, 1 = right
    )        
  )+  scale_x_discrete(labels = c("ICI-AIN", "ICI-ATN")) + 
  xlab("") + ylab("Cell Composition ratio") + ggtitle("Cell composition\nper group")
#print(p.bar1)

# p.bar2 <- niche.merge@meta.data %>% select(condition, niche_label)  %>% filter(niche_label != "Skeletal Muscle") %>% count(condition, niche_label) %>% group_by(condition) %>% mutate(ratio=n/sum(n)) %>% 
#     ggplot() + geom_bar(aes(x=condition, y=ratio, fill=niche_label), stat='identity') + scale_fill_manual(values=niche_cols) + theme_classic()+  theme(
#     legend.title = element_text(size = 6),  # Legend title size
#     legend.text  = element_text(size = 5),  # Legend labels size
#     legend.key.size = unit(0.4, "cm"),
#     legend.position='right',
#     axis.text.x = element_text(angle = 45, hjust = 1, vjust=1.05),
#   ) +xlab("")+get_theme(size=8)

#fig.size(15, 10)
#print(p.bar1/p.bar2)
p.bar1

In [ ]:
cell_counts <- obj.merge@meta.data %>% count(case_ctrl, cell_label) %>% group_by(case_ctrl) %>% mutate(ratio=n/sum(n)) %>% mutate(cell_label = factor(cell_label, levels=names(celltype_colors)))

In [ ]:
cell_counts %>% select(case_ctrl, cell_label, ratio) %>% pivot_wider(names_from=case_ctrl, values_from=ratio) %>% 
    mutate(ratio = Case / Control)

In [ ]:
cell_counts %>% select(case_ctrl, cell_label, ratio) %>% pivot_wider(names_from=case_ctrl, values_from=ratio) %>% 
    mutate(ratio = Control / Case)

In [ ]:
table(niche.merge@meta.data$niche_label)

# Fig 1d-e

In [ ]:
library(glue)
obj.cna <- association.Seurat(
    seurat_object = orig.merge.xen,
    test_var = 'case_ctrl_num',
    samplem_key = 'sample_id',
    graph_use = 'humap_fgraph',
    verbose = TRUE,
    batches = NULL, ## no batch variables to include
)

In [ ]:
dim(obj.cna)

In [ ]:
obj.cna@reductions$cna@misc$p

In [ ]:
obj.cna@reductions$cna@misc$p
p3=FeaturePlot_scCustom(obj.cna, features = c('cna_ncorrs_fdr05'), raster=T, raster.dpi=c(256, 256))[[1]] + #
    scale_color_gradient2(high = "#de2d26", mid = "#ede2d5", low = "#2c7fb8", midpoint = 0,  guide = guide_colorbar(direction = "vertical"))+
    labs(title = 'ICI-AIN-associated cells', subtitle = 'Cell filtered for local FDR<0.05', color = 'Correlation')+
    scale_x_continuous(breaks = NULL) +
    scale_y_continuous(breaks = NULL) +
    xlab("")+ylab("")+ ggplot2::theme(legend.position = "right") #+ggtitle("")
fig1e = p3

In [ ]:
p.global <- obj.cna@reductions$cna@misc$p

p.global.label <- paste0(
  "CNA global p = ",
  format.pval(p.global, digits = 3, eps = 1e-300)
)

p3 <- FeaturePlot_scCustom(
  obj.cna,
  features = c("cna_ncorrs_fdr05"),
  raster = TRUE,
  raster.dpi = c(256, 256)
)[[1]] +
  scale_color_gradient2(
    high = "#de2d26",
    mid = "#ede2d5",
    low = "#2c7fb8",
    midpoint = 0,
    guide = guide_colorbar(direction = "vertical")
  ) +
  labs(
    title = "ICI-AIN-associated cells",
    subtitle = paste0("Cell filtered for local FDR < 0.05\n", p.global.label),
    color = "Correlation"
  ) +
  scale_x_continuous(breaks = NULL) +
  scale_y_continuous(breaks = NULL) +
  xlab("") +
  ylab("") +
  theme(
    legend.position = "right",
    plot.subtitle = element_text(size = 10)
  )

fig1e <- p3

In [ ]:
head(obj.cna)

In [ ]:
fig.size(6, 8)
fig1e

# fig. 1g

In [ ]:
niche.merge@meta.data$condition_num = as.numeric(factor(str_trim(niche.merge@meta.data$condition), levels=c("Control", "Case")))

niche.cna <- association.Seurat(
    seurat_object = niche.merge,
    test_var = 'condition_num',
    samplem_key = 'sample_id',
    graph_use = 'RNA_snn',
    verbose = TRUE,
    batches = NULL, ## no batch variables to include
    #covs = c("age", "sex", "ICPi") ## no covariates to include
)

In [ ]:
p3=FeaturePlot_scCustom(niche.cna, features = c('cna_ncorrs_fdr10'), raster=T, raster.dpi = c(200, 200))[[1]] + #
    scale_color_gradient2(high = "#de2d26", mid = "white", low = "#2c7fb8", midpoint = 0,  guide = guide_colorbar(direction = "horizontal"))+
    labs(title = 'CNA disease association', subtitle = 'Filtered for FDR<0.10', color = 'Correlation')+
    scale_x_continuous(breaks = NULL) +
    scale_y_continuous(breaks = NULL) +
    xlab("")+ylab("")+ ggplot2::theme(legend.position = "right") #+ggtitle("")
fig1g = p3

In [ ]:
p.global <- niche.cna@reductions$cna@misc$p

p.global.label <- paste0(
  "CNA global p = ",
  format.pval(p.global, digits = 3, eps = 1e-300)
)

p3 <- FeaturePlot_scCustom(
  niche.cna,
  features = c("cna_ncorrs_fdr10"),
  raster = TRUE,
  raster.dpi = c(200, 200)
)[[1]] +
  scale_color_gradient2(
    high = "#de2d26",
    mid = "white",
    low = "#2c7fb8",
    midpoint = 0,
    guide = guide_colorbar(direction = "horizontal")
  ) +
  labs(
    title = "CNA disease association",
    subtitle = paste0("Filtered for local FDR < 0.10\n", p.global.label),
    color = "Correlation"
  ) +
  scale_x_continuous(breaks = NULL) +
  scale_y_continuous(breaks = NULL) +
  xlab("") +
  ylab("") +
  ggplot2::theme(
    legend.position = "right",
    plot.subtitle = element_text(size = 10)
  )

fig1g <- p3

In [ ]:
fig1g

# Fig 1f

In [ ]:
dim(lennard.subtype)

In [ ]:
dim(sc.niche$obj)

In [ ]:
sc.niche$obj@meta.data = sc.niche$obj@meta.data %>% mutate(lennard_label=lennard.subtype@meta.data$lennard_label)
sc.niche$obj@meta.data = sc.niche$obj@meta.data %>% mutate(tile_label = imm.niche@meta.data[match(sc.niche$obj@meta.data$tile_id, rownames(imm.niche@meta.data)), 'niche_label'])

In [ ]:
names(table(obj.merge@meta.data$lennard_label))

In [ ]:
table(sc.niche$obj@meta.data$lennard_label)

In [ ]:
sc.niche$obj@meta.data = sc.niche$obj@meta.data %>% mutate(cell_label=
                                                           case_when(
    lennard_label %in% c("Basophil") ~ "Mast", 
    lennard_label %in% c("Interstitial Cell") ~ "Stroma",  
    lennard_label %in% c("Collecting Duct-PC", "Collecting Duct-IC", "Connecting Tubule") ~ "Collecting Duct", 
    lennard_label %in% c("Thin Descending Limb", "Thick Ascending Limb", "Thin Ascending Limb") ~ "Loop of Henle",  
###Thin Descending Limb”, “Thick Ascending Limb” and “Thin Ascending Limb” into one cell type “Loop of Henle    
    .default = lennard_label                                                                
                                                               
                                                               ))

In [ ]:
sc.niche$obj@meta.data = sc.niche$obj@meta.data %>% mutate(tile_label=
                                                           case_when(
    tile_label %in% c("Thick Ascending Limb") ~ "Loop of Henle",  
###Thin Descending Limb”, “Thick Ascending Limb” and “Thin Ascending Limb” into one cell type “Loop of Henle    
    .default = tile_label                                                                
                                                               
                                                               ))

In [ ]:
sc.niche.heatmap<-sc.niche$obj@meta.data %>% count(tile_label, cell_label) %>% pivot_wider(names_from=cell_label, values_from=n, values_fill=0) %>% 
   filter(!is.na(tile_label))
sc.niche.heatmap = as.data.frame(sc.niche.heatmap)
rownames(sc.niche.heatmap) = sc.niche.heatmap$tile_label
sc.niche.heatmap = sc.niche.heatmap[,-1]

In [ ]:
sc.niche.heatmap <- scale(sc.niche.heatmap)

In [ ]:
length(niche_cols)

In [ ]:
length(rownames(sc.niche.heatmap))

In [ ]:
sc.niche.heatmap <- sc.niche.heatmap[rownames(sc.niche.heatmap)!="Skeletal Muscle",]

names(niche_cols) <- rownames(sc.niche.heatmap)

fig.size(5,9)
# Use niche colors here
ha1 = rowAnnotation(samples = rownames(sc.niche.heatmap),
                    col=list(samples=niche_cols),
                    annotation_name_gp = gpar(fontsize = 12),    
                    show_legend = FALSE,
                    annotation_legend_param = list(
                    samples = list(title_gp = gpar(fontsize = 5), labels_gp = gpar(fontsize = 5), direction = "horizontal")))

## Use heatmap colors

ht <- Heatmap(
  sc.niche.heatmap,
  name = "Cell Composition Ratio",
  col = heatmap_col_fun,
  width=6.5,
  height=1,
  column_names_gp = gpar(fontsize = 12),
  heatmap_legend_param = list(
    title_gp = gpar(fontsize = 5.5),
    labels_gp = gpar(fontsize = 8),
    direction = "horizontal"
  ), cluster_columns = T,
  column_names_side = "top", left_annotation = ha1)

fig1f = ht

In [ ]:
fig1f

# fig1 c-d

In [ ]:
obj.merge@meta.data = obj.merge@meta.data %>% mutate(cell_label = factor(cell_label, levels=names(celltype_colors)))

In [ ]:
# p11=DimPlot_scCustom(subset(obj.merge, subset = case_ctrl=='Case'), raster = TRUE,        # enable rasterization
#   raster.dpi = c(350, 350), group.by="cell_label", seed=99)  +ggplot2::theme(legend.position = "none",         
#                                                                              axis.line = element_line(arrow = arrow(type = "closed", length = unit(10, 'pt'))))+
#      NoLegend() + 
#     scale_x_continuous(breaks = NULL) +
#     scale_y_continuous(breaks = NULL) + NoLegend()+ggtitle("ICI-AIN")+xlab("")+ylab("")+scale_color_manual(values=celltype_colors)
# p12=DimPlot_scCustom(subset(obj.merge, subset = case_ctrl=='Control'), raster = TRUE,        # enable rasterization
#   raster.dpi = c(350, 350),  group.by="cell_label", seed=99) +ggplot2::theme(legend.position = "none") + NoLegend() + ggtitle("ICI-ATN")+scale_x_continuous(breaks = NULL)+ scale_y_continuous(breaks = NULL)+xlab("")+ylab("")+
#   scale_color_manual(values=celltype_colors)

In [ ]:
# fig1cd = p11 | p12

In [ ]:
fig1cd = DimPlot_scCustom(obj.merge, raster = TRUE,        # enable rasterization
  raster.dpi = c(256, 256), group.by="cell_label", seed=99)  +ggplot2::theme(legend.position = "bottom")+
    scale_x_continuous(breaks = NULL) +
    scale_y_continuous(breaks = NULL) + ggtitle("All cells in Xenium")+xlab("")+ylab("")+scale_color_manual(values=celltype_colors) +
    guides(color = guide_legend(nrow = 3, byrow = F,  override.aes = list(size = 5)))

In [ ]:
fig.size(5, 12)
fig1cd

# Figure f

In [ ]:
library(arrow)
library(sf)
read_parquet("shapes.parquet") %>% st_as_sf() ->etst
read_parquet("transcripts.parquet") -> transcripts

cells.labels <- obj.merge@meta.data %>% filter(orig.ident == 'output-XETG00392__0045655__BS21-N65682A2__20241025__201009_orig_seg.h5ad') %>% mutate(cell_label = factor(cell_label, levels=names(celltype_colors)))
colnames(etst)[2] = 'cell_id'

In [ ]:
# Adjust these coordinates to match your image
scalebar_x_start <- 100  # x-coordinate start
scalebar_y_start <- 10   # y-coordinate start
scalebar_length <- 200    # µm
scalebar_height <- 2     # thickness of the bar

In [ ]:
library(ggrastr)

fig.size(8, 12)
p01 = ggplot() + rasterise(
    geom_sf(data=cells.labels %>% left_join(etst, by = "cell_id"), aes(geometry=geometry, fill=cell_label), color=NA), dpi=60) + 
    scale_fill_manual(values = celltype_colors)+
  theme_void()+
  theme(panel.background = element_rect(fill = 'black')) +
  theme(aspect.ratio = 0.4, axis.line = element_line(color = 'black'), plot.background = element_blank(), panel.grid.minor = element_blank(), panel.grid.major = element_blank(), 
        legend.position='none', legend.title=element_blank()) +
  geom_rect(aes(xmin = 3100, xmax = 3350, ymin = 1600, ymax = 1800),
            fill = NA, color='blue', alpha = 1, linewidth=2, inherit.aes = FALSE) +
  geom_rect(aes(xmin = 3300, xmax = 3600, ymin = 1800, ymax = 2200),
            fill = NA, color='red', alpha = 1, linewidth=2, inherit.aes = FALSE)+
  geom_rect(aes(xmin = 2300, xmax = 2550, ymin = 1800, ymax = 2030),
            fill = NA, color='purple', alpha = 1, linewidth=2, inherit.aes = FALSE) +
  geom_segment(aes(x = scalebar_x_start,
                   y = scalebar_y_start,
                   xend = scalebar_x_start + scalebar_length,
                   yend = scalebar_y_start),
               color = "white", size = scalebar_height) +
  annotate("text", x = scalebar_x_start + scalebar_length/2, 
           y = scalebar_y_start + 135, 
           label = paste0(scalebar_length, " µm"),
           color = "white", size = 5)

In [ ]:
p01

In [ ]:
fig.size(8, 12)
p2 = ggplot() + rasterise(geom_sf(data=cells.labels %>% left_join(etst, by = "cell_id"), aes(geometry=geometry, fill=cell_label), color=NA), dpi=100) + scale_fill_manual(values = celltype_colors) + 
  geom_point_rast(data=transcripts %>% filter(qv>=20) %>% filter(feature_name %in% c('CD19', 'CD27', 'CD3E')), aes(x=x_location, y=y_location, shape=feature_name), color='white', 
             alpha=0.7, size=2.5, raster.dpi=100) + 
  coord_sf(xlim=c(3100, 3350), ylim=c(1600, 1800), expand = FALSE) + 
  theme_void()+
  theme(panel.background = element_rect(fill = 'black'), panel.border = element_rect(color = "blue", linewidth = 2, linetype = "solid", fill=NA)) +
  theme(aspect.ratio = 1, axis.line = element_line(color = 'black'), plot.background = element_blank(), panel.grid.minor = element_blank(), panel.grid.major = element_blank(),
        legend.title=element_blank(), 
        legend.position = c(0.85, 0.1), # Position legend inside (e.g., top-right)
        legend.text = element_text(size = 13, color='red', face='bold'), # Increase legend text size
  legend.key = element_rect(fill = NA, colour = NA),       # keys behind items
  legend.background = element_rect(fill = NA, colour = NA),# whole legend area
  legend.box.background = element_blank(),
    legend.spacing.x = unit(0.1, "cm"),
    legend.key.width = unit(0.3, "cm"),    
    legend.spacing.y = unit(0.1, "cm"),
    legend.key.height = unit(0.3, "cm"),                  
    legend.key.size = unit(1.5, "cm")) + guides(fill = "none") + 
  geom_segment(aes(x = 3150+scalebar_x_start,
                   y = 1598+scalebar_y_start,
                   xend = 3150+scalebar_x_start + 20,
                   yend = 1598+scalebar_y_start),
               color = "white", size = scalebar_height) +
  annotate("text", 
           x = 3150+scalebar_x_start + 20/2, 
           y = 1598+scalebar_y_start+8.5, 
           label = paste0(20, " µm"),
           color = "white", size = 5)

In [ ]:
p2

In [ ]:
fig.size(8, 12)
p12 = ggplot() + rasterise(geom_sf(data=cells.labels %>% left_join(etst, by = "cell_id"), aes(geometry=geometry, fill=cell_label), color=NA), dpi=100) + scale_fill_manual(values = celltype_colors) + 
  geom_point_rast(data=transcripts %>% filter(qv>=20) %>% filter(feature_name %in% c('PODXL', 'PLA2R1', 'CUBN')), aes(x=x_location, y=y_location, shape=feature_name), color='white',
             alpha=0.7, size=3, raster.dpi=100) + 
  coord_sf(xlim=c(1700, 2100), ylim=c(850, 1200), expand = FALSE) + 
  theme_void()+
  theme(panel.background = element_rect(fill = 'black'), panel.border = element_rect(color = "red", linewidth = 2, linetype = "solid", fill=NA)) +
  theme(aspect.ratio = 1, axis.line = element_line(color = 'black'), plot.background = element_blank(), panel.grid.minor = element_blank(), panel.grid.major = element_blank(),
        legend.title=element_blank(), 
        legend.position = c(0.85, 0.1), # Position legend inside (e.g., top-right)
        legend.text = element_text(size = 13, color='red', face='bold'), # Increase legend text size
  legend.key = element_rect(fill = NA, colour = NA),       # keys behind items
  legend.background = element_rect(fill = NA, colour = NA),# whole legend area
  legend.box.background = element_blank(),
    # horizontal spacing: pull text closer to symbols
    legend.spacing.x = unit(0.1, "cm"),
    legend.key.width = unit(0.3, "cm"),    
    # vertical spacing: stack items closer
    legend.spacing.y = unit(0.1, "cm"),
    legend.key.height = unit(0.3, "cm"),
        legend.key.size = unit(1.5, "cm")) + guides(fill = "none") + geom_segment(aes(x = 1830 + scalebar_x_start,
                   y = 864+scalebar_y_start,
                   xend = 1830+scalebar_x_start + 20,
                   yend = 864+scalebar_y_start),
               color = "white", size = scalebar_height) +
  annotate("text", x = 1830+scalebar_x_start + 20/2, 
           y = 864+scalebar_y_start+13,
           label = paste0(20, " µm"),
           color = "white", size = 5)

In [ ]:
p12

In [ ]:
fig.size(8, 12)
p13 = ggplot() + rasterize(geom_sf(data=cells.labels %>% left_join(etst, by = "cell_id"), aes(geometry=geometry, fill=cell_label), color=NA), dpi=100) + scale_fill_manual(values = celltype_colors) + 
  geom_point_rast(data=transcripts %>% filter(qv>=20) %>% filter(feature_name %in% c('COL5A1', 'TNXB', 'CXCL12')), aes(x=x_location, y=y_location, shape=feature_name), color='white',
             alpha=0.7, size=3, raster.dpi=100) + 
  coord_sf(xlim=c(2300, 2550), ylim=c(1800, 2030), expand = FALSE) + 
  theme_void()+
  theme(panel.background = element_rect(fill = 'black'), panel.border = element_rect(color = "purple", linewidth = 2, linetype = "solid", fill=NA)) +
  theme(aspect.ratio = 1, axis.line = element_line(color = 'black'), plot.background = element_blank(), panel.grid.minor = element_blank(), panel.grid.major = element_blank(),
        legend.title=element_blank(), 
        legend.position = c(0.16, 0.9), # Position legend inside (e.g., top-right)
        legend.text = element_text(size = 13, color='red', face='bold'), # Increase legend text size
  legend.key = element_rect(fill = NA, colour = NA),       # keys behind items
  legend.background = element_rect(fill = NA, colour = NA),# whole legend area
  legend.box.background = element_blank(),
    legend.spacing.x = unit(0.1, "cm"),
    legend.key.width = unit(0.3, "cm"),    
    # vertical spacing: stack items closer
    legend.spacing.y = unit(0.1, "cm"),
    legend.key.height = unit(0.3, "cm"),        
        legend.key.size = unit(1.5, "cm")) + guides(fill = "none")+ geom_segment(aes(x = 2300 + scalebar_x_start,
                   y = 2000+scalebar_y_start,
                   xend = 2300+scalebar_x_start + 20,
                   yend = 2000+scalebar_y_start),
               color = "white", size = scalebar_height) +
  annotate("text", x = 2300+scalebar_x_start + 20/2, 
           y = 2000+scalebar_y_start+8.5, 
           label = paste0(20, " µm"),
           color = "white", size = 5)
p13

In [ ]:
library(patchwork)
fig.size(12, 25)
fig1f = (p01 | p13 | p2 | p12) + plot_layout(ncol=4)
#fig1f = p01 | p13

# Assemble figure 1

In [ ]:
library(patchwork)
fig.size(20.5, 20.5)

pdf("Fig1_v4.pdf", width=18, height=18.5)
((plot_spacer()+fig1b+plot_layout(ncol=2, widths=c(1.3, 5)))) / (fig1cd|fig1e|p.bar1) / (fig1f) + plot_layout(nrow=3, heights=c(5, 5, 5.5)) + 
 plot_annotation(
    tag_levels = list(c("B", "C", "D", "E", "F", "G")), 
 ) & theme(plot.tag = element_text(face = "bold", size = 26))
dev.off()

In [ ]:
# library(patchwork)
# fig.size(20.5, 20.5)
# pdf("Fig1_v4.pdf", width=16.5, height=16.5)
# ((plot_spacer()+fig1b+plot_layout(ncol=2, widths=c(1.3, 5)))) / (fig1cd|fig1e|p.bar1) + plot_layout(nrow=2, heights=c(5, 5, 5.5)) + 
#  plot_annotation(
#     tag_levels = list(c("B", "C", "D", "E", "F", "G")), 
#  ) & theme(plot.tag = element_text(face = "bold", size = 26))
# dev.off()

In [ ]:
# #ggsave("Fig1_v3.pdf", width=18.5, height=18.5)
# ggsave("Fig1_v4.pdf", width=16.5, height=16.5, dpi=120)

In [ ]:
getwd()